## Baic model(Not the final)
This is just the Baseline model with no hyperparameter tuning or any other  optimisation techniques purely trined to observe the improvement 

In [3]:
pip install -q transformers accelerate

In [4]:
 from google.colab import files
uploaded = files.upload()
DATA_PATH = "mtsamples.csv"


Saving mtsamples.csv to mtsamples.csv


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# Imports
# ══════════════════════════════════════════════════════════════════════════
import re, warnings, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


In [6]:
# ══════════════════════════════════════════════════════════════════════════
# Load & preprocess
# ══════════════════════════════════════════════════════════════════════════
df = pd.read_csv(DATA_PATH)
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

df["medical_specialty"] = df["medical_specialty"].str.strip()

counts = df["medical_specialty"].value_counts()
df["label"] = df["medical_specialty"].apply(
    lambda x: x if counts[x] >= 20 else "Other"
)

df["text"] = (
    df["transcription"].fillna("") + " " + df["description"].fillna("")
).str.strip()

df = df[df["text"].str.len() > 30].reset_index(drop=True)

print(f"Rows   : {len(df)}")
print(f"Classes: {df['label'].nunique()}")
print(df["label"].value_counts().to_string())

Rows   : 4987
Classes: 30
label
Surgery                          1098
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  228
Neurology                         223
SOAP / Chart / Progress Notes     166
Obstetrics / Gynecology           160
Urology                           157
Other                             125
Discharge Summary                 108
ENT - Otolaryngology               96
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    62
Psychiatry / Psychology            53
Office Notes                       50
Podiatry                           47
Dermatology                        29
Dentistry         

In [7]:

# ══════════════════════════════════════════════════════════════════════════
# Label encode + stratified split  (70 / 15 / 15)
# ══════════════════════════════════════════════════════════════════════════
le = LabelEncoder()
y  = le.fit_transform(df["label"].values)
X  = df["text"].values
NUM_CLASSES = len(le.classes_)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print(f"Train:{len(X_train)}  Val:{len(X_val)}  Test:{len(X_test)}")


Train:3490  Val:748  Test:749


In [8]:

# ══════════════════════════════════════════════════════════════════════════
# Tokeniser + Dataset
# ══════════════════════════════════════════════════════════════════════════
MODEL_NAME = "roberta-base"
MAX_LEN    = 512
BATCH_SIZE = 16

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

class MedDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            list(texts),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : self.enc["input_ids"][idx],
            "attention_mask" : self.enc["attention_mask"][idx],
            "labels"         : self.labels[idx],
        }

train_dl = DataLoader(MedDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(MedDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(MedDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("Dataloaders ready ")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataloaders ready 


In [9]:
# ══════════════════════════════════════════════════════════════════════════
# Model + optimiser
# ══════════════════════════════════════════════════════════════════════════
EPOCHS  = 5
LR      = 2e-5
WARMUP  = 0.1

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)
model.to(device)

total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(total_steps * WARMUP)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

label_counts  = np.bincount(y_train)
class_weights = torch.tensor(
    len(y_train) / (NUM_CLASSES * label_counts), dtype=torch.float
).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

print(f"Model loaded |  Total steps: {total_steps}  |  Warmup: {warmup_steps}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded |  Total steps: 1095  |  Warmup: 109


In [10]:
# ══════════════════════════════════════════════════════════════════════════
# Training loop
# ══════════════════════════════════════════════════════════════════════════
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labs = batch["labels"].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = loss_fn(logits, labs)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item()
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    mf1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), acc, mf1

best_val_f1 = 0.0
history     = []

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(train_dl, train=True)
    vl_loss, vl_acc, vl_f1 = run_epoch(val_dl,   train=False)
    history.append(dict(epoch=epoch,
                        tr_loss=tr_loss, tr_acc=tr_acc, tr_f1=tr_f1,
                        vl_loss=vl_loss, vl_acc=vl_acc, vl_f1=vl_f1))
    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train loss={tr_loss:.4f} acc={tr_acc:.4f} F1={tr_f1:.4f} | "
          f"Val   loss={vl_loss:.4f} acc={vl_acc:.4f} F1={vl_f1:.4f}")
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), "/content/roberta_best.pt")
        print(f"  Saved best checkpoint (val F1={best_val_f1:.4f})")

Epoch 1/5 | Train loss=3.4100 acc=0.0384 F1=0.0120 | Val   loss=3.3977 acc=0.2206 F1=0.0120
  Saved best checkpoint (val F1=0.0120)
Epoch 2/5 | Train loss=3.4039 acc=0.0619 F1=0.0253 | Val   loss=3.3950 acc=0.0735 F1=0.0046
Epoch 3/5 | Train loss=3.4019 acc=0.0550 F1=0.0208 | Val   loss=3.3943 acc=0.0735 F1=0.0046
Epoch 4/5 | Train loss=3.3996 acc=0.0885 F1=0.0233 | Val   loss=3.3936 acc=0.0735 F1=0.0046
Epoch 5/5 | Train loss=3.3979 acc=0.1103 F1=0.0260 | Val   loss=3.3933 acc=0.2206 F1=0.0121
  Saved best checkpoint (val F1=0.0121)


In [11]:

# ══════════════════════════════════════════════════════════════════════════
# Test evaluation
# ══════════════════════════════════════════════════════════════════════════
model.load_state_dict(torch.load("/content/roberta_best.pt"))
_, test_acc, test_f1 = run_epoch(test_dl, train=False)
print(f"\n{'='*55}")
print(f"TEST Accuracy : {test_acc:.4f}")
print(f"TEST Macro-F1 : {test_f1:.4f}")
print(f"{'='*55}")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_dl:
        logits = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        ).logits
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print(classification_report(all_labels, all_preds,
                             target_names=le.classes_, zero_division=0))



TEST Accuracy : 0.2203
TEST Macro-F1 : 0.0120
                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.00      0.00      0.00        56
   Consult - History and Phy.       0.00      0.00      0.00        78
   Cosmetic / Plastic Surgery       0.00      0.00      0.00         4
                    Dentistry       0.00      0.00      0.00         4
                  Dermatology       0.00      0.00      0.00         4
            Discharge Summary       0.00      0.00      0.00        16
         ENT - Otolaryngology       0.00      0.00      0.00        15
       Emergency Room Reports       0.00      0.00      0.00        11
             Gastroenterology       0.00      0.00      0.00        34
             General Medicine       0.00      0.00      0.00        39
        Hematology - Oncology       0.00      0.00      0.00        14
                      Letters       0.00      0.00      0.00         3
                   Nephrology